In [1]:
import torch
print(torch.__version__)

2.11.0+cu128


In [3]:
from google.colab import files
uploaded = files.upload()

Saving daphnet+freezing+of+gait.zip to daphnet+freezing+of+gait.zip


In [4]:
import zipfile

with zipfile.ZipFile("daphnet+freezing+of+gait.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [12]:
import os
os.listdir()

['.config',
 'daphnet+freezing+of+gait.zip',
 'preprocess.py',
 'artifacts',
 '__pycache__',
 'dataset_fog_release',
 '.ipynb_checkpoints',
 'sample_data']

In [29]:
import os
import json
import random
from datetime import datetime

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import lightgbm as lgb

import shap

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

from sklearn.calibration import (
    CalibratedClassifierCV
)

from preprocess import load_and_preprocess, WINDOW_SIZE, STRIDE

In [14]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", DEVICE)

DATA_PATH = "dataset"

ARTIFACT_DIR = "artifacts"

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True
)

Using: cuda


In [15]:
DATA_PATH = "/content/dataset_fog_release/dataset"
(
    train_loader,
    val_loader,
    test_loader,
    X_train_lgbm,
    X_val_lgbm,
    X_test_lgbm,
    y_train,
    y_val,
    y_test,
    scaler,
    class_weights,
    groups_train,
    groups_val,
    groups_test
) = load_and_preprocess(DATA_PATH)

Patients Found : 17

Dataset Statistics
Total Windows : 29940
Patients      : 10
Class 0: 12045
Class 1: 17895
✓ No patient leakage detected.
Train: (21284, 128, 9)
Validation: (2691, 128, 9)
Test: (5965, 128, 9)


In [16]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm1d(
            out_channels
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm1d(
            out_channels
        )

        if in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1
                ),
                nn.BatchNorm1d(
                    out_channels
                )
            )

        else:

            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = torch.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity

        return torch.relu(out)


class FOGModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            ResidualBlock(9,32),

            nn.MaxPool1d(2),

            nn.Dropout(0.20),

            ResidualBlock(32,64),

            nn.MaxPool1d(2),

            nn.Dropout(0.25),

            ResidualBlock(64,128),

            nn.MaxPool1d(2)

        )

        self.lstm = nn.LSTM(

            input_size=128,

            hidden_size=128,

            num_layers=2,

            bidirectional=True,

            dropout=0.30,

            batch_first=True

        )

        self.attention = nn.Sequential(

            nn.Linear(
                256,
                128
            ),

            nn.Tanh(),

            nn.Linear(
                128,
                1
            )

        )

        self.classifier = nn.Sequential(

            nn.Dropout(0.40),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(
                128,
                2
            )

        )

    def forward(self,x):

        x = self.features(x)

        x = x.permute(
            0,
            2,
            1
        )

        outputs,_ = self.lstm(x)

        weights = torch.softmax(

            self.attention(outputs),

            dim=1

        )

        context = torch.sum(

            weights * outputs,

            dim=1

        )

        return self.classifier(context)

In [17]:
model = FOGModel().to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

print(model)

FOGModel(
  (features): Sequential(
    (0): ResidualBlock(
      (conv1): Conv1d(9, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential(
        (0): Conv1d(9, 32, kernel_size=(1,), stride=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Dropout(p=0.2, inplace=False)
    (3): ResidualBlock(
      (conv1): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNo

In [18]:
from sklearn.metrics import (accuracy_score,roc_auc_score)

scaler_amp = torch.amp.GradScaler("cuda")

EPOCHS = 75
PATIENCE = 12

best_auc = 0.0
best_epoch = 0
patience_counter = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "val_auc": []
}

print("=" * 70)
print("Starting CNN-LSTM Training")
print("=" * 70)

for epoch in range(EPOCHS):

    ###############################
    # TRAIN
    ###############################
    model.train()
    running_loss = 0.0

    train_preds = []
    train_labels = []

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast(
            device_type="cuda"
        ):

            outputs = model(x_batch)

            loss = criterion(
                outputs,
                y_batch
            )

        scaler_amp.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler_amp.step(
            optimizer
        )

        scaler_amp.update()

        running_loss += (
            loss.item() *
            x_batch.size(0)
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        train_preds.extend(
            preds.cpu().numpy()
        )

        train_labels.extend(
            y_batch.cpu().numpy()
        )

    train_loss = (
        running_loss /
        len(train_loader.dataset)
    )

    train_acc = accuracy_score(
        train_labels,
        train_preds
    )

    ##################################
    # VALIDATION
    ##################################

    model.eval()

    running_loss = 0.0

    val_probs = []

    val_preds = []

    val_labels = []

    with torch.no_grad():

        for x_batch, y_batch in val_loader:

            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            with torch.amp.autocast(
                device_type="cuda"
            ):

                outputs = model(
                    x_batch
                )

                loss = criterion(
                    outputs,
                    y_batch
                )

            probs = torch.softmax(
                outputs,
                dim=1
            )[:,1]

            preds = torch.argmax(
                outputs,
                dim=1
            )

            running_loss += (
                loss.item() *
                x_batch.size(0)
            )

            val_probs.extend(
                probs.cpu().numpy()
            )

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_labels.extend(
                y_batch.cpu().numpy()
            )

    val_loss = (
        running_loss /
        len(val_loader.dataset)
    )

    val_acc = accuracy_score(
        val_labels,
        val_preds
    )

    val_auc = roc_auc_score(
        val_labels,
        val_probs
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["train_acc"].append(
        train_acc
    )

    history["val_acc"].append(
        val_acc
    )

    history["val_auc"].append(
        val_auc
    )

    scheduler.step(
        val_auc
    )

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

    ##################################
    # EARLY STOPPING
    ##################################

    if val_auc > best_auc:

        best_auc = val_auc
        best_epoch = epoch + 1

        patience_counter = 0

        torch.save(
            model.state_dict(),
            os.path.join(
                ARTIFACT_DIR,
                "gait_model.pt"
            )
        )

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print()
        print(
            f"Early stopping at epoch {epoch+1}"
        )

        break

print()
print("=" * 70)
print(
    f"Best Validation AUC : {best_auc:.4f}"
)
print(
    f"Best Epoch : {best_epoch}"
)
print("=" * 70)

Starting CNN-LSTM Training
Epoch 01/75 | Train Loss: 0.3303 | Val Loss: 0.6270 | Train Acc: 0.8601 | Val Acc: 0.6997 | Val AUC: 0.8410
Epoch 02/75 | Train Loss: 0.2912 | Val Loss: 0.5879 | Train Acc: 0.8809 | Val Acc: 0.7187 | Val AUC: 0.8305
Epoch 03/75 | Train Loss: 0.2774 | Val Loss: 0.5253 | Train Acc: 0.8837 | Val Acc: 0.7651 | Val AUC: 0.8258
Epoch 04/75 | Train Loss: 0.2561 | Val Loss: 0.4664 | Train Acc: 0.8924 | Val Acc: 0.7830 | Val AUC: 0.8084
Epoch 05/75 | Train Loss: 0.2387 | Val Loss: 0.5127 | Train Acc: 0.9015 | Val Acc: 0.6994 | Val AUC: 0.8376
Epoch 06/75 | Train Loss: 0.2237 | Val Loss: 0.5161 | Train Acc: 0.9060 | Val Acc: 0.7737 | Val AUC: 0.8431
Epoch 07/75 | Train Loss: 0.2250 | Val Loss: 0.5394 | Train Acc: 0.9064 | Val Acc: 0.7603 | Val AUC: 0.8138
Epoch 08/75 | Train Loss: 0.2199 | Val Loss: 0.5469 | Train Acc: 0.9098 | Val Acc: 0.6916 | Val AUC: 0.8205
Epoch 09/75 | Train Loss: 0.2200 | Val Loss: 0.6453 | Train Acc: 0.9112 | Val Acc: 0.6276 | Val AUC: 0.8306
E

In [19]:
model = FOGModel().to(DEVICE)

model.load_state_dict(
    torch.load(
        os.path.join(
            ARTIFACT_DIR,
            "gait_model.pt"
        ),
        map_location=DEVICE
    )
)

model.eval()

print("Best model loaded.")

Best model loaded.


In [20]:
from sklearn.metrics import (
    f1_score,
    roc_auc_score
)

val_probs = []
val_labels = []

with torch.no_grad():

    for x_batch, y_batch in val_loader:

        x_batch = x_batch.to(DEVICE)

        outputs = model(x_batch)

        probs = torch.softmax(
            outputs,
            dim=1
        )[:,1]

        val_probs.extend(
            probs.cpu().numpy()
        )

        val_labels.extend(
            y_batch.numpy()
        )

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

best_threshold = 0.5
best_f1 = 0

for threshold in thresholds:

    preds = (
        np.array(val_probs)
        >= threshold
    ).astype(int)

    score = f1_score(
        val_labels,
        preds
    )

    if score > best_f1:

        best_f1 = score
        best_threshold = threshold

print()

print("Best Threshold :", best_threshold)

print("Validation F1 :", best_f1)

joblib.dump(
    best_threshold,
    os.path.join(
        ARTIFACT_DIR,
        "decision_threshold.pkl"
    )
)


Best Threshold : 0.15000000000000002
Validation F1 : 0.8732832339984452


['artifacts/decision_threshold.pkl']

In [21]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    accuracy_score,
    average_precision_score
)

test_probs = []
test_labels = []

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(DEVICE)

        outputs = model(x_batch)

        probs = torch.softmax(
            outputs,
            dim=1
        )[:,1]

        test_probs.extend(
            probs.cpu().numpy()
        )

        test_labels.extend(
            y_batch.numpy()
        )

test_preds = (
    np.array(test_probs)
    >= best_threshold
).astype(int)

accuracy = accuracy_score(
    test_labels,
    test_preds
)

precision = precision_score(
    test_labels,
    test_preds
)

recall = recall_score(
    test_labels,
    test_preds
)

f1 = f1_score(
    test_labels,
    test_preds
)

auc = roc_auc_score(
    test_labels,
    test_probs
)

pr_auc = average_precision_score(
    test_labels,
    test_probs
)

cm = confusion_matrix(
    test_labels,
    test_preds
)

print()

print(classification_report(
    test_labels,
    test_preds
))

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)
print("ROC AUC  :", auc)
print("PR AUC   :", pr_auc)

print()

print(cm)


              precision    recall  f1-score   support

           0       0.88      0.54      0.67      2546
           1       0.74      0.95      0.83      3419

    accuracy                           0.77      5965
   macro avg       0.81      0.74      0.75      5965
weighted avg       0.80      0.77      0.76      5965

Accuracy : 0.7741827326068734
Precision: 0.7351339083068543
Recall   : 0.947353027200936
F1       : 0.8278594249201278
ROC AUC  : 0.8526461456667341
PR AUC   : 0.8664430617951517

[[1379 1167]
 [ 180 3239]]


In [22]:
confidence = np.maximum(
    test_probs,
    1 - np.array(test_probs)
)

print()

print("Average Confidence")

print(np.mean(confidence))


Average Confidence
0.85170215


In [23]:
joblib.dump(
    confidence,
    os.path.join(
        ARTIFACT_DIR,
        "prediction_confidence.pkl"
    )
)

['artifacts/prediction_confidence.pkl']

In [30]:
background = []

for x_batch, _ in train_loader:
    background.append(x_batch)
    if len(background) == 5:
        break

background = torch.cat(background)

In [35]:
model_cpu = FOGModel()
model_cpu.load_state_dict(
    torch.load(
        os.path.join(ARTIFACT_DIR, "gait_model.pt"),
        map_location="cpu"
    )
)

model_cpu.eval()

FOGModel(
  (features): Sequential(
    (0): ResidualBlock(
      (conv1): Conv1d(9, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential(
        (0): Conv1d(9, 32, kernel_size=(1,), stride=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Dropout(p=0.2, inplace=False)
    (3): ResidualBlock(
      (conv1): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNo

In [34]:
import shap

model_cpu.eval()

explainer = shap.GradientExplainer(
    model_cpu,
    background
)

print("SHAP explainer created.")

SHAP explainer created.


In [36]:
import json
from datetime import datetime

metadata = {

    "model_id":"gait_cnn_lstm_v1",

    "modality":"gait",

    "architecture":"Residual CNN + BiLSTM + Attention",

    "framework":"PyTorch",

    "model_version":"v1",

    "validation_auc":float(best_auc),

    "validation_f1":float(best_f1),

    "decision_threshold":float(best_threshold),

    "supports_uncertainty":True,

    "uncertainty_method":"softmax_probability",

    "explainability":"SHAP GradientExplainer",

    "window_size":WINDOW_SIZE,

    "stride":STRIDE,

    "created_at":datetime.now().isoformat()

}

with open(

    os.path.join(
        ARTIFACT_DIR,
        "metadata.json"
    ),

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print(metadata)

{'model_id': 'gait_cnn_lstm_v1', 'modality': 'gait', 'architecture': 'Residual CNN + BiLSTM + Attention', 'framework': 'PyTorch', 'model_version': 'v1', 'validation_auc': 0.8430872931531805, 'validation_f1': 0.8732832339984452, 'decision_threshold': 0.15000000000000002, 'supports_uncertainty': True, 'uncertainty_method': 'softmax_probability', 'explainability': 'SHAP GradientExplainer', 'window_size': 128, 'stride': 64, 'created_at': '2026-06-25T10:56:12.090783'}


In [37]:
joblib.dump(

    best_auc,

    os.path.join(
        ARTIFACT_DIR,
        "validation_auc.pkl"
    )

)

joblib.dump(

    best_f1,

    os.path.join(
        ARTIFACT_DIR,
        "validation_f1.pkl"
    )

)

joblib.dump(

    history,

    os.path.join(
        ARTIFACT_DIR,
        "training_history.pkl"
    )

)

print("Artifacts saved successfully.")

Artifacts saved successfully.


In [38]:
print("="*60)

print("Generated Artifacts")

print("="*60)

for file in sorted(os.listdir(ARTIFACT_DIR)):

    print(file)

Generated Artifacts
class_weights.pkl
decision_threshold.pkl
gait_model.pt
metadata.json
prediction_confidence.pkl
preprocessing_config.pkl
scaler.pkl
training_history.pkl
validation_auc.pkl
validation_f1.pkl
